In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import expit
from sklearn.base import BaseEstimator, ClassifierMixin, RegressorMixin

Las dos clases heredan de `BaseEstimator` y de `RegressorMixin` o `ClassifierMixin` de `scikit-learn`. Con eso, y con los hiperparámetros guardados en `__init__` con el mismo nombre que los argumentos, funcionan dentro de un `Pipeline` y con `GridSearchCV` (ejercicios 3 y 4): no hace falta programar la validación cruzada a mano. `RegressorMixin` agrega el método `score` ($R^2$) y `ClassifierMixin`, el accuracy.

## Ejercicio 1

**Costo de Elastic Net** (el bias $\theta_0$ no se penaliza):

$$
J(\theta_0,\theta)=\frac{1}{2n}\|y-\theta_0-X\theta\|^2+\lambda_1\|\theta\|_1+\frac{\lambda_2}{2}\|\theta\|_2^2 .
$$

**Descenso por coordenadas.** Se actualiza una coordenada por vez, dejando las demás fijas. Para la coordenada $j$, con el residuo parcial $r^{(j)}=y-\theta_0-\sum_{k\ne j}x_k\theta_k$ (sin la contribución de la variable $j$), el problema en $\theta_j$ es unidimensional y tiene solución cerrada:

$$
\theta_j=\frac{S(\rho_j,\lambda_1)}{z_j+\lambda_2},\qquad
\rho_j=\frac1nx_j^\top r^{(j)},\qquad
z_j=\frac1n\|x_j\|^2,\qquad
S(a,\lambda)=\operatorname{sign}(a)\max(|a|-\lambda,0).
$$

$S$ es el **umbral suave**: si $|\rho_j|\le\lambda_1$, la coordenada queda exactamente en 0. El bias se actualiza como la media del residuo, $\theta_0=\frac1n\sum_i(y_i-x_i^\top\theta)$.

**Implementación eficiente**: se mantiene el residuo completo $r=y-\theta_0-X\theta$. Como $r^{(j)}=r+x_j\theta_j$, resulta $\rho_j=\frac1nx_j^\top r+z_j\theta_j$, y después de cambiar $\theta_j$ alcanza con corregir $r$ en $x_j\,\Delta\theta_j$.

**Criterio de parada**: una pasada por todas las coordenadas (y el bias) en la que el mayor cambio $\max_j|\Delta\theta_j|$ es menor que `tol`.

**Equivalencia con `scikit-learn`**: `ElasticNet` minimiza el mismo costo con `alpha` $=\lambda_1+\lambda_2$ y `l1_ratio` $=\lambda_1/(\lambda_1+\lambda_2)$.

In [ ]:
class myElasticNet(RegressorMixin, BaseEstimator):
    """
    Regresión lineal con penalización Elastic Net, ajustada por descenso por coordenadas.
    """
    def __init__(self, lambda1=1.0, lambda2=0.0, max_iter=1000, tol=1e-6, ajustar_bias=True):
        """
        Inicializa el modelo.

        Parámetros:
        - lambda1: intensidad de la penalización l1 (0: Ridge)
        - lambda2: intensidad de la penalización l2 (0: Lasso)
        - max_iter: número máximo de pasadas por todas las coordenadas
        - tol: tolerancia sobre el mayor cambio de una coordenada en una pasada
        - ajustar_bias: si es False, el modelo no tiene bias (θ0 = 0)
        """

        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.max_iter = max_iter
        self.tol = tol
        self.ajustar_bias = ajustar_bias

    def umbral_suave(self, a, umbral):
        """
        S(a, umbral) = sign(a) max(|a| - umbral, 0).

        Parámetros:
        - a: float o array
        - umbral: float >= 0

        Retorna:
        - float o array, de la forma de a
        """

        # TODO

        return ...

    def costo(self, X, y, theta0, theta):
        """
        Costo de Elastic Net.

        Parámetros:
        - X: (n, p), y: (n,), theta0: float, theta: (p,)

        Retorna:
        - J: float
        """

        # TODO: error cuadrático / (2n) + λ1 ||θ||_1 + λ2/2 ||θ||²

        r = ...  # (n,)
        return ...

    def fit(self, X, y):
        """
        Ajusta el modelo por descenso por coordenadas, partiendo de θ = 0.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        # TODO:
        # - Inicializar θ = 0, θ0 = media de y (o 0 si no hay bias) y el residuo r
        # - En cada pasada: actualizar el bias y luego cada coordenada con el umbral suave,
        #   corrigiendo el residuo; registrar el mayor cambio
        # - Parar si el mayor cambio es menor que tol; si se llega a max_iter, avisar

        n, p = X.shape
        theta = np.zeros(p)  # (p,)
        theta0 = np.mean(y) if self.ajustar_bias else 0.0
        r = ...  # residuo (n,)
        z = ...  # ||x_j||² / n, (p,)
        cost_hist = [self.costo(X, y, theta0, theta)]

        for t in range(self.max_iter):
            mayor_cambio = 0.0

            if self.ajustar_bias:
                nuevo = ...  # θ0 que anula la media del residuo
                r -= ...  # (n,)
                mayor_cambio = abs(nuevo - theta0)
                theta0 = nuevo

            for j in range(p):
                anterior = theta[j]
                rho = ...  # x_j^T r^(j) / n
                theta[j] = ...
                r -= ...  # (n,)
                mayor_cambio = max(mayor_cambio, abs(theta[j] - anterior))

            cost_hist.append(self.costo(X, y, theta0, theta))
            if mayor_cambio < self.tol:
                break
        else:
            warnings.warn(f"myElasticNet no convergió en {self.max_iter} pasadas")

        self.theta0 = theta0
        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = t + 1
        return self

    def predict(self, X):
        """
        Predicciones del modelo.

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: (n,)
        """

        # TODO

        return ...  # (n,)

##### Solución

In [ ]:
class myElasticNet(RegressorMixin, BaseEstimator):
    """
    Regresión lineal con penalización Elastic Net, ajustada por descenso por coordenadas.
    """
    def __init__(self, lambda1=1.0, lambda2=0.0, max_iter=1000, tol=1e-6, ajustar_bias=True):
        """
        Inicializa el modelo.

        Parámetros:
        - lambda1: intensidad de la penalización l1 (0: Ridge)
        - lambda2: intensidad de la penalización l2 (0: Lasso)
        - max_iter: número máximo de pasadas por todas las coordenadas
        - tol: tolerancia sobre el mayor cambio de una coordenada en una pasada
        - ajustar_bias: si es False, el modelo no tiene bias (θ0 = 0)
        """

        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.max_iter = max_iter
        self.tol = tol
        self.ajustar_bias = ajustar_bias

    def umbral_suave(self, a, umbral):
        """
        S(a, umbral) = sign(a) max(|a| - umbral, 0).

        Parámetros:
        - a: float o array
        - umbral: float >= 0

        Retorna:
        - float o array, de la forma de a
        """

        return np.sign(a) * np.maximum(np.abs(a) - umbral, 0)

    def costo(self, X, y, theta0, theta):
        """
        Costo de Elastic Net.

        Parámetros:
        - X: (n, p), y: (n,), theta0: float, theta: (p,)

        Retorna:
        - J: float
        """

        r = y - theta0 - X @ theta  # (n,)
        return r @ r / (2 * len(y)) + self.lambda1 * np.sum(np.abs(theta)) + self.lambda2 / 2 * theta @ theta

    def fit(self, X, y):
        """
        Ajusta el modelo por descenso por coordenadas, partiendo de θ = 0.

        Parámetros:
        - X: (n, p)
        - y: (n,)

        Retorna:
        - self
        """

        n, p = X.shape
        theta = np.zeros(p)  # (p,)
        theta0 = np.mean(y) if self.ajustar_bias else 0.0
        r = y - theta0 - X @ theta  # residuo (n,)
        z = np.sum(X**2, axis=0) / n  # ||x_j||² / n, (p,)
        cost_hist = [self.costo(X, y, theta0, theta)]

        for t in range(self.max_iter):
            mayor_cambio = 0.0

            if self.ajustar_bias:
                nuevo = theta0 + np.mean(r)  # θ0 que anula la media del residuo
                r -= nuevo - theta0  # (n,)
                mayor_cambio = abs(nuevo - theta0)
                theta0 = nuevo

            for j in range(p):
                anterior = theta[j]
                rho = X[:, j] @ r / n + z[j] * anterior  # x_j^T r^(j) / n
                theta[j] = self.umbral_suave(rho, self.lambda1) / (z[j] + self.lambda2)
                r -= X[:, j] * (theta[j] - anterior)  # (n,)
                mayor_cambio = max(mayor_cambio, abs(theta[j] - anterior))

            cost_hist.append(self.costo(X, y, theta0, theta))
            if mayor_cambio < self.tol:
                break
        else:
            warnings.warn(f"myElasticNet no convergió en {self.max_iter} pasadas")

        self.theta0 = theta0
        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = t + 1
        return self

    def predict(self, X):
        """
        Predicciones del modelo.

        Parámetros:
        - X: (n, p)

        Retorna:
        - y_pred: (n,)
        """

        return self.theta0 + X @ self.theta  # (n,)

##### Verificación con scikit-learn

In [ ]:
from sklearn.linear_model import ElasticNet

rng = np.random.default_rng(0)
X_sim = rng.normal(size=(200, 5))  # (n, p)
y_sim = 1.0 + X_sim @ np.array([2.0, -1.0, 0.0, 0.0, 0.5]) + rng.normal(0, 0.5, 200)

for lambda1, lambda2 in [(0.1, 0.0), (0.05, 0.05), (0.0, 0.1)]:
    modelo = myElasticNet(lambda1, lambda2).fit(X_sim, y_sim)
    alpha = lambda1 + lambda2
    sk = ElasticNet(alpha=alpha, l1_ratio=lambda1 / alpha, tol=1e-10).fit(X_sim, y_sim)
    print(
        f"λ1 = {lambda1}, λ2 = {lambda2}: {modelo.n_iter_} pasadas, θ = {modelo.theta.round(4)}, "
        f"máx. dif. con sklearn = {np.abs(modelo.theta - sk.coef_).max():.1e}"
    )

---

## Ejercicio 2

**Costo**, con $s=\theta_0+X\theta$ y el bias sin penalizar:

$$
J(\theta_0,\theta)=\underbrace{\frac1n\sum_{i=1}^n\big[\log(1+e^{s_i})-y_is_i\big]+\frac{\lambda_2}{2}\|\theta\|^2}_{f\ \text{(diferenciable)}}+\lambda_1\|\theta\|_1 .
$$

**Gradiente proximal.** La norma $\ell_1$ no es derivable en 0, así que no se puede aplicar descenso por gradiente directamente. El método del gradiente proximal da un paso de gradiente sobre la parte diferenciable $f$ y después aplica el umbral suave, que es el operador proximal de $\eta\lambda_1\|\cdot\|_1$:

$$
\theta_0\leftarrow\theta_0-\eta\,\frac{\partial f}{\partial\theta_0},\qquad
\theta\leftarrow S\big(\theta-\eta\,\nabla_\theta f,\ \eta\lambda_1\big),
\qquad
\frac{\partial f}{\partial\theta_0}=\frac1n\sum_i(\pi_i-y_i),\quad
\nabla_\theta f=\frac1nX^\top(\pi-y)+\lambda_2\theta .
$$

Es el mismo umbral suave del descenso por coordenadas, y por eso también deja coeficientes exactamente en 0. Converge si $\eta\le1/L$, con $L=\lambda_{\max}(X^\top X)/(4n)+\lambda_2$ una cota de la curvatura de $f$; si no se indica $\eta$, la clase usa $1/L$.

**Criterio de parada**: $\max_j|\Delta\theta_j|<$ `tol` entre dos iteraciones (incluyendo el bias).

**Equivalencia con `scikit-learn`**: `LogisticRegression(C=..., l1_ratio=..., solver="saga")` minimiza el mismo costo con `C` $=1/\big(n(\lambda_1+\lambda_2)\big)$ y `l1_ratio` $=\lambda_1/(\lambda_1+\lambda_2)$.

In [ ]:
class myLogisticRegressionElasticNet(ClassifierMixin, BaseEstimator):
    """
    Regresión logística con penalización Elastic Net, ajustada por gradiente proximal.
    """
    def __init__(self, lambda1=0.01, lambda2=0.0, eta=None, max_iter=20000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - lambda1: intensidad de la penalización l1
        - lambda2: intensidad de la penalización l2
        - eta: tasa de aprendizaje; si es None, se usa 1/L
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre el mayor cambio de un coeficiente
        """

        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.eta = eta
        self.max_iter = max_iter
        self.tol = tol

    def umbral_suave(self, a, umbral):
        """
        S(a, umbral) = sign(a) max(|a| - umbral, 0).

        Parámetros:
        - a: float o array
        - umbral: float >= 0

        Retorna:
        - float o array, de la forma de a
        """
        return np.sign(a) * np.maximum(np.abs(a) - umbral, 0)

    def costo(self, X, y, theta0, theta):
        """
        Costo logístico + λ1 ||θ||_1 + λ2/2 ||θ||².

        Parámetros:
        - X: (n, p), y: (n,), theta0: float, theta: (p,)

        Retorna:
        - J: float
        """

        # TODO: costo logístico estable con np.logaddexp más las dos penalizaciones

        s = ...  # (n,)
        return ...

    def fit(self, X, y):
        """
        Ajusta el modelo por gradiente proximal, partiendo de θ = 0.

        Parámetros:
        - X: (n, p)
        - y: (n,), con valores 0 y 1

        Retorna:
        - self
        """

        # TODO:
        # - η = 1/L si no se indicó
        # - En cada iteración: paso de gradiente sobre f para θ0 y θ; umbral suave en θ
        # - Parar si el mayor cambio es menor que tol; si se llega a max_iter, avisar

        n, p = X.shape
        self.classes_ = np.unique(y)  # lo usan algunas funciones de scikit-learn
        eta = self.eta
        if eta is None:
            eta = ...
        theta0, theta = 0.0, np.zeros(p)
        cost_hist = [self.costo(X, y, theta0, theta)]

        for t in range(self.max_iter):
            error = ...  # π - y, (n,)
            nuevo0 = ...
            nuevo = ...  # (p,)
            mayor_cambio = max(abs(nuevo0 - theta0), np.max(np.abs(nuevo - theta)))
            theta0, theta = nuevo0, nuevo
            cost_hist.append(self.costo(X, y, theta0, theta))
            if mayor_cambio < self.tol:
                break
        else:
            warnings.warn(f"myLogisticRegressionElasticNet no convergió en {self.max_iter} iteraciones")

        self.theta0 = theta0
        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = t + 1
        return self

    def predict_proba(self, X):
        """
        Probabilidad estimada de la clase 1.

        Parámetros:
        - X: (n, p)

        Retorna:
        - proba: (n,)
        """

        # TODO

        return ...  # (n,)

    def predict(self, X, umbral=0.5):
        """
        Etiqueta predicha: 1 si la probabilidad supera el umbral.

        Parámetros:
        - X: (n, p)
        - umbral: float en (0, 1)

        Retorna:
        - y_pred: (n,)
        """

        # TODO

        return ...  # (n,)

##### Solución

In [ ]:
class myLogisticRegressionElasticNet(ClassifierMixin, BaseEstimator):
    """
    Regresión logística con penalización Elastic Net, ajustada por gradiente proximal.
    """
    def __init__(self, lambda1=0.01, lambda2=0.0, eta=None, max_iter=20000, tol=1e-6):
        """
        Inicializa el modelo.

        Parámetros:
        - lambda1: intensidad de la penalización l1
        - lambda2: intensidad de la penalización l2
        - eta: tasa de aprendizaje; si es None, se usa 1/L
        - max_iter: número máximo de iteraciones
        - tol: tolerancia sobre el mayor cambio de un coeficiente
        """

        self.lambda1 = lambda1
        self.lambda2 = lambda2
        self.eta = eta
        self.max_iter = max_iter
        self.tol = tol

    def umbral_suave(self, a, umbral):
        """
        S(a, umbral) = sign(a) max(|a| - umbral, 0).

        Parámetros:
        - a: float o array
        - umbral: float >= 0

        Retorna:
        - float o array, de la forma de a
        """
        return np.sign(a) * np.maximum(np.abs(a) - umbral, 0)

    def costo(self, X, y, theta0, theta):
        """
        Costo logístico + λ1 ||θ||_1 + λ2/2 ||θ||².

        Parámetros:
        - X: (n, p), y: (n,), theta0: float, theta: (p,)

        Retorna:
        - J: float
        """

        s = theta0 + X @ theta  # (n,)
        return (
            np.mean(np.logaddexp(0, s) - y * s)
            + self.lambda1 * np.sum(np.abs(theta))
            + self.lambda2 / 2 * theta @ theta
        )

    def fit(self, X, y):
        """
        Ajusta el modelo por gradiente proximal, partiendo de θ = 0.

        Parámetros:
        - X: (n, p)
        - y: (n,), con valores 0 y 1

        Retorna:
        - self
        """

        n, p = X.shape
        self.classes_ = np.unique(y)  # lo usan algunas funciones de scikit-learn
        eta = self.eta
        if eta is None:
            eta = 1 / (np.linalg.eigvalsh(X.T @ X).max() / (4 * n) + self.lambda2)
        theta0, theta = 0.0, np.zeros(p)
        cost_hist = [self.costo(X, y, theta0, theta)]

        for t in range(self.max_iter):
            error = expit(theta0 + X @ theta) - y  # π - y, (n,)
            nuevo0 = theta0 - eta * np.mean(error)
            nuevo = self.umbral_suave(theta - eta * (X.T @ error / n + self.lambda2 * theta), eta * self.lambda1)  # (p,)
            mayor_cambio = max(abs(nuevo0 - theta0), np.max(np.abs(nuevo - theta)))
            theta0, theta = nuevo0, nuevo
            cost_hist.append(self.costo(X, y, theta0, theta))
            if mayor_cambio < self.tol:
                break
        else:
            warnings.warn(f"myLogisticRegressionElasticNet no convergió en {self.max_iter} iteraciones")

        self.theta0 = theta0
        self.theta = theta
        self.cost_hist = cost_hist
        self.n_iter_ = t + 1
        return self

    def predict_proba(self, X):
        """
        Probabilidad estimada de la clase 1.

        Parámetros:
        - X: (n, p)

        Retorna:
        - proba: (n,)
        """

        return expit(self.theta0 + X @ self.theta)  # (n,)

    def predict(self, X, umbral=0.5):
        """
        Etiqueta predicha: 1 si la probabilidad supera el umbral.

        Parámetros:
        - X: (n, p)
        - umbral: float en (0, 1)

        Retorna:
        - y_pred: (n,)
        """

        return (self.predict_proba(X) > umbral).astype(int)  # (n,)

##### Verificación con scikit-learn

In [ ]:
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)
n = 400
X_sim = rng.normal(size=(n, 6))  # (n, p)
y_sim = rng.binomial(1, expit(0.5 + X_sim @ np.array([2.0, -1.5, 0.0, 0.0, 0.0, 1.0])))

lambda1, lambda2 = 0.02, 0.01
modelo = myLogisticRegressionElasticNet(lambda1, lambda2, tol=1e-8).fit(X_sim, y_sim)
sk = LogisticRegression(
    C=1 / (n * (lambda1 + lambda2)),
    l1_ratio=lambda1 / (lambda1 + lambda2),
    solver="saga",
    max_iter=100000,
    tol=1e-10,
).fit(X_sim, y_sim)

print(f"Iteraciones: {modelo.n_iter_}")
print("myLogisticRegressionElasticNet:", modelo.theta.round(4))
print("sklearn (saga):", sk.coef_[0].round(4))
print(f"Máx. diferencia: {np.abs(modelo.theta - sk.coef_[0]).max():.1e}")
print("Coeficientes en 0:", np.sum(modelo.theta == 0), "propios,", np.sum(sk.coef_ == 0), "de sklearn")

#### *(Adicional)* Subgradiente vs. gradiente proximal

Un **subgradiente** de $\|\theta\|_1$ es $\operatorname{sign}(\theta)$ (con $\operatorname{sign}(0)=0$). El descenso por subgradiente usa $\theta\leftarrow\theta-\eta\big(\nabla_\theta f+\lambda_1\operatorname{sign}(\theta)\big)$: es lo más parecido a un descenso por gradiente, pero con paso fijo no converge (oscila alrededor del óptimo) y casi nunca deja coeficientes exactamente en 0.

In [ ]:
def subgradiente_logistica_en(X, y, lambda1, lambda2, eta, max_iter):
    """
    Regresión logística con Elastic Net por descenso por subgradiente, con paso fijo.

    Parámetros:
    - X: (n, p), y: (n,)
    - lambda1, lambda2: penalizaciones
    - eta: tasa de aprendizaje
    - max_iter: cantidad de iteraciones (sin criterio de parada)

    Retorna:
    - theta0: float
    - theta: (p,)
    """

    # TODO: como el gradiente proximal, pero sumando λ1 sign(θ) al gradiente en lugar del umbral suave

    n, p = X.shape
    theta0, theta = 0.0, np.zeros(p)
    for t in range(max_iter):
        error = ...  # (n,)
        theta0 = ...
        theta = ...  # (p,)
    return theta0, theta

##### Solución

In [ ]:
def subgradiente_logistica_en(X, y, lambda1, lambda2, eta, max_iter):
    """
    Regresión logística con Elastic Net por descenso por subgradiente, con paso fijo.

    Parámetros:
    - X: (n, p), y: (n,)
    - lambda1, lambda2: penalizaciones
    - eta: tasa de aprendizaje
    - max_iter: cantidad de iteraciones (sin criterio de parada)

    Retorna:
    - theta0: float
    - theta: (p,)
    """

    n, p = X.shape
    theta0, theta = 0.0, np.zeros(p)
    for t in range(max_iter):
        error = expit(theta0 + X @ theta) - y  # (n,)
        theta0 = theta0 - eta * np.mean(error)
        theta = theta - eta * (X.T @ error / n + lambda2 * theta + lambda1 * np.sign(theta))  # (p,)
    return theta0, theta

##### Comparación

In [ ]:
eta = 1 / (np.linalg.eigvalsh(X_sim.T @ X_sim).max() / (4 * n) + lambda2)
theta0_sub, theta_sub = subgradiente_logistica_en(X_sim, y_sim, lambda1, lambda2, eta, modelo.n_iter_)

print("Proximal:", modelo.theta.round(5), f"costo = {modelo.cost_hist[-1]:.6f}")
print("Subgradiente:", theta_sub.round(5), f"costo = {modelo.costo(X_sim, y_sim, theta0_sub, theta_sub):.6f}")
print("Coeficientes exactamente en 0: proximal", np.sum(modelo.theta == 0), "| subgradiente", np.sum(theta_sub == 0))

---

## Ejercicio 3

Mismos datos y partición que en la guía 2.2 (sin el escalado de `load_diabetes`, 80/20, `random_state=42`), para poder comparar con la regresión sin regularización. Estandarizar es imprescindible antes de penalizar: la penalización trata igual a todos los coeficientes, así que tienen que estar en la misma escala.

In [ ]:
from sklearn.datasets import load_diabetes
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

diabetes = load_diabetes(scaled=False)
X_train, X_test, y_train, y_test = train_test_split(
    diabetes.data,
    diabetes.target,
    test_size=0.2,
    random_state=42,
)
nombres = diabetes.feature_names
print("Train:", X_train.shape, "| Test:", X_test.shape)

#### a) Camino de regularización

In [ ]:
escalador = StandardScaler().fit(X_train)
X_train_s = escalador.transform(X_train)

lambdas_lasso = np.logspace(-2, 1.5, 40)
lambdas_ridge = np.logspace(-2, 3, 40)
camino_lasso = np.array([myElasticNet(lam, 0.0).fit(X_train_s, y_train).theta for lam in lambdas_lasso])  # (40, p)
camino_ridge = np.array([myElasticNet(0.0, lam).fit(X_train_s, y_train).theta for lam in lambdas_ridge])  # (40, p)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)
for ax, lambdas, camino, titulo, simbolo in [
    (axes[0], lambdas_lasso, camino_lasso, "Lasso", "$\\lambda_1$"),
    (axes[1], lambdas_ridge, camino_ridge, "Ridge", "$\\lambda_2$"),
]:
    for j, nombre in enumerate(nombres):
        ax.plot(lambdas, camino[:, j], label=nombre)
    ax.set_xscale("log")
    ax.axhline(0, color="k", linewidth=0.5)
    ax.set_xlabel(simbolo)
    ax.set_title(titulo)
axes[0].set_ylabel("coeficiente")
axes[1].legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

# Orden en que las variables se vuelven no nulas en Lasso (de λ1 grande a chico)
entrada = {nombre: lambdas_lasso[camino_lasso[:, j] != 0].max() for j, nombre in enumerate(nombres) if np.any(camino_lasso[:, j] != 0)}
print("Lasso, mayor λ1 con coeficiente no nulo:")
for nombre, lam in sorted(entrada.items(), key=lambda kv: -kv[1]):
    print(f"  {nombre}: {lam:.2f}")

**Para discutir:** ¿En qué orden entran las variables en el camino de Lasso? ¿Qué diferencia hay con el de Ridge?

#### b) Selección de $\lambda_1$ y $\lambda_2$ por validación cruzada

El escalador va dentro del `Pipeline`: en cada partición de la validación cruzada se ajusta solo con los datos de entrenamiento de esa partición. Los nombres de los parámetros llevan el nombre del paso (`myelasticnet__lambda1`).

In [ ]:
grilla = {
    "myelasticnet__lambda1": [0.0, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0],
    "myelasticnet__lambda2": [0.0, 0.01, 0.1, 1.0],
}
busqueda = GridSearchCV(
    make_pipeline(StandardScaler(), myElasticNet()),
    grilla,
    cv=5,
    scoring="neg_mean_squared_error",
).fit(X_train, y_train)

print("Mejores hiperparámetros:", busqueda.best_params_)
print(f"MSE de validación cruzada: {-busqueda.best_score_:.2f}")
print(f"MSE en test: {mean_squared_error(y_test, busqueda.predict(X_test)):.2f}")

resultados = pd.DataFrame(busqueda.cv_results_)
tabla = resultados.pivot_table(
    index="param_myelasticnet__lambda1",
    columns="param_myelasticnet__lambda2",
    values="mean_test_score",
)
(-tabla).round(1)

#### c) Comparación con la regresión sin regularización

In [ ]:
sin_reg = make_pipeline(StandardScaler(), myElasticNet(0.0, 0.0)).fit(X_train, y_train)
mejor = busqueda.best_estimator_

print(f"MSE en test sin regularización: {mean_squared_error(y_test, sin_reg.predict(X_test)):.2f}")
print(f"MSE en test con regularización: {mean_squared_error(y_test, mejor.predict(X_test)):.2f}")

pd.DataFrame(
    {"sin regularización": sin_reg[-1].theta, "regularizado": mejor[-1].theta},
    index=nombres,
).round(2)

#### d) Comparación con `ElasticNet` de `scikit-learn`

In [ ]:
lambda1 = busqueda.best_params_["myelasticnet__lambda1"]
lambda2 = busqueda.best_params_["myelasticnet__lambda2"]
alpha = lambda1 + lambda2

sk = make_pipeline(
    StandardScaler(),
    ElasticNet(alpha=alpha, l1_ratio=lambda1 / alpha, tol=1e-10),
).fit(X_train, y_train)

print(f"λ1 = {lambda1}, λ2 = {lambda2} -> alpha = {alpha}, l1_ratio = {lambda1 / alpha}")
print(f"Máx. diferencia de coeficientes: {np.abs(mejor[-1].theta - sk[-1].coef_).max():.1e}")
print(f"MSE en test: propio {mean_squared_error(y_test, mejor.predict(X_test)):.4f}, sklearn {mean_squared_error(y_test, sk.predict(X_test)):.4f}")

#### e) *(Adicional)* Lasso y *basis pursuit*

Con $A$ de forma $(m,p)$, $m<p$, y $b=Ax^\star$ sin ruido, Lasso sin bias minimiza $\frac1{2m}\|b-Ax\|^2+\lambda_1\|x\|_1$: cuando $\lambda_1\to0$, el término cuadrático obliga a cumplir $Ax=b$ y, entre esas soluciones, gana la de menor $\|x\|_1$, que es *basis pursuit*.

In [ ]:
import cvxpy as cp

rng = np.random.default_rng(0)
m, p = 40, 100
A = rng.normal(0, 1 / np.sqrt(m), size=(m, p))
x_star = np.zeros(p)
x_star[rng.choice(p, 5, replace=False)] = rng.normal(size=5)
b = A @ x_star

x = cp.Variable(p)
cp.Problem(cp.Minimize(cp.norm1(x)), [A @ x == b]).solve()
x_bp = x.value
print(f"Basis pursuit: error relativo = {np.linalg.norm(x_bp - x_star) / np.linalg.norm(x_star):.1e}")

for lam in [0.01, 0.001, 0.0001]:
    lasso = myElasticNet(lam, 0.0, max_iter=100000, tol=1e-10, ajustar_bias=False).fit(A, b)
    print(
        f"Lasso, λ1 = {lam}: ||x - x_BP|| / ||x_BP|| = {np.linalg.norm(lasso.theta - x_bp) / np.linalg.norm(x_bp):.1e}, "
        f"||Ax - b|| = {np.linalg.norm(A @ lasso.theta - b):.1e}, no nulos = {np.sum(lasso.theta != 0)}"
    )